In [1]:
import sys
print(sys.executable)

C:\Users\ghuy2\Documents\BACHKHOA\HK_252\BigData\kafka\kafka-lab\Scripts\python.exe


In [2]:
import os

os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"

In [3]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("MyApp")
    .master("local[*]")
    .config("spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .getOrCreate()
)

C:\Users\ghuy2\Documents\BACHKHOA\HK_252\BigData\kafka\kafka-lab\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies  = spark.read.csv(path + "/movies.csv",  header=True, inferSchema=True)
df_tags    = spark.read.csv(path + "/tags.csv",    header=True, inferSchema=True)

df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 1 row

+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
+-------+----------------+--------------------+
only showing top 1 row

+------+-------+-----+----------+
|userId|movieId|  tag| timestamp|
+------+-------+-----+----------+
|     2|  60756|funny|1445714994|
+------+-------+-----+----------+
only showing top 1 row



In [5]:
admin_client = AdminClient({'bootstrap.servers': 'localhost:9092'})

# Delete old topics if they exist
admin_client.delete_topics(['ratings', 'movies', 'tags'], operation_timeout=10)
print("Deleted if exists")

# Create 3 topics (3 replicas to match 3 brokers)
new_topics = [
    NewTopic(topic="ratings", num_partitions=1, replication_factor=3),
    NewTopic(topic="movies",  num_partitions=1, replication_factor=3),
    NewTopic(topic="tags",    num_partitions=1, replication_factor=3),
]
for topic, f in admin_client.create_topics(new_topics).items():
    try:
        f.result()
        print(f"Topic {topic} created")
    except Exception as e:
        print(f"Failed to create topic {topic}: {e}")

Deleted if exists
Topic ratings created
Topic movies created
Topic tags created


In [6]:
for df, topic in [(df_ratings, "ratings"), (df_movies, "movies"), (df_tags, "tags")]:
    (df.selectExpr("to_json(struct(*)) AS value")
       .write
       .format("kafka")
       .option("kafka.bootstrap.servers", "localhost:9092")
       .option("topic", topic)
       .save())

# Verify — binary check
df_check = (spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "ratings")
    .load()
    .selectExpr("CAST(value AS STRING)"))
df_check.show(1)

+--------------------+
|               value|
+--------------------+
|{"userId":1,"movi...|
+--------------------+
only showing top 1 row



In [7]:
spark.read.format("kafka")

In [8]:
md = admin_client.list_topics(timeout=10)

print("Brokers:", md.brokers)
print("Topics:", md.topics)

Brokers: {1: BrokerMetadata(1, localhost:9092), 2: BrokerMetadata(2, localhost:9192), 3: BrokerMetadata(3, localhost:9292)}
Topics: {'movies': TopicMetadata(movies, 1 partitions), 'ratings': TopicMetadata(ratings, 1 partitions), 'tags': TopicMetadata(tags, 1 partitions)}
